# Module 18 — Testing, Debugging, and Quality

## Exercise 18.2 — The same service, two test suites

The point of this exercise is the experiment at the bottom, not the tests
themselves. Run it before reading further.
Run:  pytest ex02_doubles.py -v

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. pytest, the parts you need

In [ ]:
def test_addition() -> None:
    assert add(2, 3) == 5          # plain assert; pytest rewrites it to show
                                    # both operands on failure

```bash
pytest                     # run everything
pytest -q                  # quiet
pytest -x                  # stop at the first failure
pytest -k "user and not slow"     # select by name
pytest -m integration      # select by marker
pytest --lf                # last failed
pytest --ff                # failed first, then the rest
pytest -vv                 # full diff on assertion failure
pytest --pdb               # drop into the debugger on failure
pytest -n auto             # parallel (pytest-xdist)
```

`--lf` and `--ff` are the two that change your day: after a broad failure, you
iterate on only the failing tests until they pass.

### Exceptions and approximations

In [ ]:
import pytest

def test_raises() -> None:
    with pytest.raises(ValueError, match="must be positive"):
        parse(-1)

def test_exception_attributes() -> None:
    with pytest.raises(ValidationError) as exc_info:
        validate({"age": -1})
    assert exc_info.value.field == "age"      # Module 16: carry data

def test_float() -> None:
    assert compute() == pytest.approx(0.3)    # Module 03

Always pass `match=`. `pytest.raises(ValueError)` alone passes when *any*
`ValueError` is raised, including one from a typo in your test setup.

---

## Concept 2. Fixtures

In [ ]:
@pytest.fixture
def db() -> Iterator[Database]:
    conn = Database(":memory:")
    conn.migrate()
    yield conn              # the test runs here
    conn.close()            # teardown, even if the test fails

def test_insert(db: Database) -> None:
    db.insert({"id": 1})
    assert db.count() == 1

A fixture is dependency injection for tests (Module 12), and it composes:
fixtures can request other fixtures.

```text
@pytest.fixture(scope="session")     # once per test run
@pytest.fixture(scope="module")      # once per test file
@pytest.fixture(scope="function")    # the default: once per test
```


**Scope is a trade between speed and isolation, and getting it wrong is the
most common cause of "passes alone, fails in the suite".** A session-scoped
fixture holding mutable state is shared by every test; one test mutating it
changes what the others see, and the failure depends on *ordering*. Session
scope is for genuinely immutable or externally-managed things: a started
container, a compiled asset, a read-only fixture file.

### The built-in fixtures worth knowing

In [ ]:
def test_files(tmp_path: Path) -> None:          # a fresh temp dir per test
    (tmp_path / "f.txt").write_text("x", encoding="utf-8")

def test_env(monkeypatch: pytest.MonkeyPatch) -> None:
    monkeypatch.setenv("API_KEY", "test")        # undone automatically
    monkeypatch.setattr(module, "CONSTANT", 42)
    monkeypatch.chdir(tmp_path)

def test_output(capsys: pytest.CaptureFixture[str]) -> None:
    run()
    assert "done" in capsys.readouterr().out

def test_logs(caplog: pytest.LogCaptureFixture) -> None:
    with caplog.at_level(logging.WARNING):
        risky()
    assert "retrying" in caplog.text

`monkeypatch` undoes everything at teardown, which `setattr` by hand does not.

### `conftest.py`

Fixtures defined there are available to every test in that directory and below,
with no import. Put shared fixtures there; put nothing else there, because code
in `conftest.py` is invisible to a reader of the test file.

---

## Concept 4. Test doubles, and why fakes beat mocks

| Kind | What it is | Use when |
|---|---|---|
| **Dummy** | A placeholder never used | Filling a required parameter |
| **Stub** | Returns canned answers | You need a specific return value |
| **Fake** | A working, simpler implementation | **The default choice** |
| **Spy** | Records how it was called | You must assert on an interaction |
| **Mock** | A spy with pre-set expectations | Rarely |

In [ ]:
# a FAKE: a real implementation, in memory
class InMemoryUserRepo:
    def __init__(self) -> None:
        self._users: dict[int, User] = {}
    def save(self, user: User) -> None:
        self._users[user.id] = user
    def get(self, uid: int) -> User | None:
        return self._users.get(uid)

def test_registration_stores_the_user() -> None:
    repo = InMemoryUserRepo()
    service = RegistrationService(repo)
    service.register("ada@example.com")
    assert repo.get(1).email == "ada@example.com"     # asserts an OUTCOME

versus

In [ ]:
def test_registration_calls_save() -> None:
    repo = Mock()
    RegistrationService(repo).register("ada@example.com")
    repo.save.assert_called_once()                     # asserts an INTERACTION

The mock version passes if `save` is called **with the wrong user**. It also
breaks the moment you rename `save` or call it twice for a good reason. A mock
test is coupled to the implementation; a fake test is coupled to the behaviour.

**Use a mock only when the interaction *is* the requirement** — "an email was
sent", "the audit log recorded it", "the payment gateway was called exactly
once".

### Patching: where, not what

In [ ]:
# app/service.py
from app.clients import fetch_user      # a COPY of the reference (Module 06)

# test
patch("app.clients.fetch_user")         # WRONG: service.py's copy is unaffected
patch("app.service.fetch_user")         # RIGHT: patches the name in use

**Patch where the name is used, not where it is defined.** This is Module 06's
`from x import y` binding rule, and it accounts for a large share of "the patch
did nothing" confusion.

Better still: do not patch. If the dependency is injected (Module 12), you pass
a fake and no patching is needed. **Heavy patching is a design smell** — it is
usually telling you the code constructs its own dependencies.

---

## Concept 5. Coverage, and what it does not tell you

```bash
pytest --cov=src --cov-report=term-missing
pytest --cov=src --cov-branch          # branch coverage: much more honest
```

Coverage tells you which lines *ran*. It does not tell you whether anything was
*asserted*:

In [ ]:
def test_nothing() -> None:
    process_everything()        # 100% coverage, zero assertions, always passes

Use coverage to **find untested code**, never as a quality target. A number
target produces tests written to raise the number, which are worse than no
tests because they take time to run and give false confidence.

Branch coverage is worth enabling: a line with `if x:` counts as covered when
only the true branch ever ran.

**Where to look in a coverage report:** error-handling paths (usually the least
covered and the most dangerous), boundary conditions, and any file with high
coverage and few assertions.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: pytest, the parts you need
- Section 2: Fixtures
- Section 3: `parametrize`
- Section 4: Test doubles, and why fakes beat mocks
- Section 5: Coverage, and what it does not tell you
- Section 6: Property-based testing
- Section 7: Debugging
- Section 8: Linting and formatting
- Section 9: Designing for testability

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from typing import Protocol
from unittest.mock import Mock

import pytest

---

## `User`

_User_

In [ ]:
@dataclass(frozen=True)
class User:
    id: int
    email: str
    created: datetime

---

## `UserRepository`

_UserRepository_

In [ ]:
class UserRepository(Protocol):
    def save(self, user: User) -> None: ...
    def get(self, uid: int) -> User | None: ...
    def next_id(self) -> int: ...

---

## `Mailer`

_Mailer_

In [ ]:
class Mailer(Protocol):
    def send(self, to: str, subject: str, body: str) -> None: ...

---

## `RegistrationService`

_RegistrationService_

In [ ]:
class RegistrationService:
    def __init__(self, repo: UserRepository, mailer: Mailer,
                 now=datetime.now) -> None:  # type: ignore[no-untyped-def]
        self._repo = repo
        self._mailer = mailer
        self._now = now

    def register(self, email: str) -> User:
        if "@" not in email:
            raise ValueError(f"not an email address: {email!r}")
        user = User(id=self._repo.next_id(), email=email, created=self._now())
        self._repo.save(user)
        self._mailer.send(email, "Welcome", f"Hello {email}")
        return user

---

## `InMemoryUserRepo`

A working implementation. Implement save, get, next_id.

In [ ]:
class InMemoryUserRepo:
    """A working implementation. Implement save, get, next_id."""

---

## `RecordingMailer`

A spy: record (to, subject, body) tuples in .sent.

In [ ]:
class RecordingMailer:
    """A spy: record (to, subject, body) tuples in .sent."""

---

## `TestWithMocks`

_TestWithMocks_

In [ ]:
class TestWithMocks:
    def test_saves_the_user(self) -> None:
        repo = Mock()
        repo.next_id.return_value = 1
        service = RegistrationService(repo, Mock())
        service.register("ada@example.com")
        repo.save.assert_called_once()

    def test_sends_an_email(self) -> None:
        mailer = Mock()
        repo = Mock()
        repo.next_id.return_value = 1
        RegistrationService(repo, mailer).register("ada@example.com")
        mailer.send.assert_called_once()

---

## `TestWithFakes`

_TestWithFakes_

In [ ]:
class TestWithFakes:
    def test_saves_the_user(self) -> None:
        """Assert the OUTCOME: the user is retrievable afterwards, with the
        right email and the injected timestamp."""

    def test_sends_an_email(self) -> None:
        """Assert the outcome: an email was recorded, addressed to the right
        person, with the right content."""

---

## `test_created_timestamp_is_exact`

`now` is injected. Write a test asserting the EXACT timestamp, and note

In [ ]:
def test_created_timestamp_is_exact() -> None:
    """`now` is injected. Write a test asserting the EXACT timestamp, and note
    that this test cannot become flaky at midnight or across a DST change --
    which a test using the real clock can."""

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    raise SystemExit(pytest.main([__file__, "-v"]))

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.